# Simultaneous equations

In [2]:
import os
import polars as pl
import numpy as np

from scipy.optimize import minimize
import statsmodels.api as sm  # for linear regression

import matplotlib.pyplot as plt

## Load data

Patient demographics by MSOA:

In [3]:
path_to_msoa_stats = os.path.join('data', 'msoa_cleaned.csv')

df_stats = pl.read_csv(path_to_msoa_stats)

In [4]:
df_stats.head()

MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64
"""Adur 001""",14.333333,16.924833,8815,"""E""",6799,1251,474,0.79763,0.146762,0.055608,"""E02006534""",0.0559,0.0528,0.0422,0.7872,0.062
"""Adur 002""",7.333333,6.4704,7263,"""E""",5537,838,259,0.83464,0.126319,0.039041,"""E02006535""",0.0578,0.0774,0.0492,0.7467,0.0692
"""Adur 003""",9.333333,13.7334,7354,"""E""",5820,969,311,0.819718,0.136479,0.043803,"""E02006536""",0.0609,0.0582,0.0421,0.7729,0.0661
"""Adur 004""",21.0,26.199857,10582,"""E""",7872,1546,709,0.777328,0.152661,0.070011,"""E02006537""",0.0465,0.0438,0.0367,0.8091,0.0638
"""Adur 005""",13.666667,11.7948,9059,"""E""",7106,1081,339,0.833451,0.126789,0.039761,"""E02006538""",0.0597,0.067,0.0425,0.7643,0.0662


Check sum of admissions for Welsh areas:

In [5]:
df_stats.filter(df_stats['country'] == 'W')['admissions'].sum()

0.0

Welsh data is always zero so remove it.

In [6]:
df_stats = df_stats.filter(df_stats['country'] != 'W')

Recalculate total numbers of patients:

In [7]:
df_stats = df_stats.with_columns((pl.col('good_health') + pl.col('fair health') + pl.col('bad health')).alias('total_health'))

In [8]:
df_stats.head()

MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64
"""Adur 001""",14.333333,16.924833,8815,"""E""",6799,1251,474,0.79763,0.146762,0.055608,"""E02006534""",0.0559,0.0528,0.0422,0.7872,0.062,8524
"""Adur 002""",7.333333,6.4704,7263,"""E""",5537,838,259,0.83464,0.126319,0.039041,"""E02006535""",0.0578,0.0774,0.0492,0.7467,0.0692,6634
"""Adur 003""",9.333333,13.7334,7354,"""E""",5820,969,311,0.819718,0.136479,0.043803,"""E02006536""",0.0609,0.0582,0.0421,0.7729,0.0661,7100
"""Adur 004""",21.0,26.199857,10582,"""E""",7872,1546,709,0.777328,0.152661,0.070011,"""E02006537""",0.0465,0.0438,0.0367,0.8091,0.0638,10127
"""Adur 005""",13.666667,11.7948,9059,"""E""",7106,1081,339,0.833451,0.126789,0.039761,"""E02006538""",0.0597,0.067,0.0425,0.7643,0.0662,8526


Pick out column names for the health and age proportions:

In [9]:
health_numbers = ['good_health', 'fair health', 'bad health']
props_health = ['prop_good_health', 'prop_fair health', 'prop_bad health']
props_age = [
    'age_less65_proportion', 'age_65_proportion', 'age_70_proportion',
    'age_75_proportion', 'age_over80_proportion'
]

Calculate numbers of patients in each age band:

In [10]:
age_numbers = []

for col in props_age:
    new_col = col.replace('_proportion', '')
    age_numbers.append(new_col)
    df_stats = df_stats.with_columns((pl.col(col) * pl.col('total_health')).alias(new_col))

In [11]:
df_stats[['total_health'] + props_age + age_numbers].head()

total_health,age_less65_proportion,age_65_proportion,age_70_proportion,age_75_proportion,age_over80_proportion,age_less65,age_65,age_70,age_75,age_over80
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
8524,0.7872,0.0559,0.0528,0.0422,0.062,6710.0928,476.4916,450.0672,359.7128,528.488
6634,0.7467,0.0578,0.0774,0.0492,0.0692,4953.6078,383.4452,513.4716,326.3928,459.0728
7100,0.7729,0.0609,0.0582,0.0421,0.0661,5487.59,432.39,413.22,298.91,469.31
10127,0.8091,0.0465,0.0438,0.0367,0.0638,8193.7557,470.9055,443.5626,371.6609,646.1026
8526,0.7643,0.0597,0.067,0.0425,0.0662,6516.4218,509.0022,571.242,362.355,564.4212


## Set up bins for IMD scores

Use quantiles so that each bin contains 10% of the MSOA (to match plot notebook).

The resulting dictionary has keys for which quantile it is and values for the left side (smaller, minimum value in bin) of the IMD bin.

In [12]:
dict_quantiles = {}

for q in np.arange(0.0, 1.01, 0.2):
    v = df_stats['IMD2019Score'].quantile(q)
    # Flip the ranking because lower rank (more deprived) should be for
    # higher IMD scores (more deprived).
    q_store = 1.0 - q
    dict_quantiles[round(q_store, 1)] = round(v, 5)

In [13]:
dict_quantiles

{1.0: 2.2122,
 0.8: 10.369,
 0.6: 15.2564,
 0.4: 21.81475,
 0.2: 31.84075,
 0.0: 87.02675}

Pick out just the values for the left edges of the bins:

In [14]:
imd_bin_left_edges = {}
for k in list(dict_quantiles.keys())[:-1]:
    imd_bin_left_edges[k] = dict_quantiles[k]

In [15]:
imd_bin_left_edges

{1.0: 2.2122, 0.8: 10.369, 0.6: 15.2564, 0.4: 21.81475, 0.2: 31.84075}

Place MSOA into these bins:

In [16]:
# Make columns for the results with a placeholder value:
# df_stats = df_stats.with_columns(pl.lit(0).alias('imd_bin_min'))
# df_stats = df_stats.with_columns(pl.lit(0).alias('imd_bin_max'))
df_stats = df_stats.with_columns(pl.lit(0.0).alias('depriv_quantile_min'))
df_stats = df_stats.with_columns(pl.lit(0.0).alias('depriv_quantile_max'))

for q, quantile in enumerate(list(dict_quantiles.keys())[:-1]):
    # Pick out the bin edges:
    qmin = list(dict_quantiles.keys())[q+1]
    q0 = dict_quantiles[quantile]
    q1 = dict_quantiles[qmin]
    # Find a mask for the demographic data that contains
    # only MSOA with IMD scores in this bin.
    mask = (df_stats['IMD2019Score'] >= q0) & (df_stats['IMD2019Score'] < q1)
    if q == len(dict_quantiles) - 2:
        # Also allow values at the right edge of the final bin.
        mask = mask | (df_stats['IMD2019Score'] == q1)

    # Update the bin min/max values for these rows:
    df_stats = df_stats.with_columns(
        pl.when((mask))
        .then(qmin)         # replace with bin min
        .otherwise(pl.col('depriv_quantile_min'))  # otherwise keep the existing value
        .name.keep()
    )
    df_stats = df_stats.with_columns(
        pl.when((mask))
        .then(quantile)         # replace with bin min
        .otherwise(pl.col('depriv_quantile_max'))  # otherwise keep the existing value
        .name.keep()
    )

Check that data was binned correctly:

In [17]:
df_stats[['IMD2019Score', 'depriv_quantile_min', 'depriv_quantile_max']]

IMD2019Score,depriv_quantile_min,depriv_quantile_max
f64,f64,f64
16.924833,0.4,0.6
6.4704,0.8,1.0
13.7334,0.6,0.8
26.199857,0.2,0.4
11.7948,0.6,0.8
…,…,…
3.25925,0.8,1.0
7.29475,0.8,1.0
12.117,0.6,0.8


## Group LSOAs

In [21]:
qmin = 0.0

df_here = df_stats.filter(df_stats['depriv_quantile_min'] == qmin)

# Randomly assign groups:
groups = np.zeros(len(df_here), dtype=int)
inds_available = np.arange(len(df_here))

n_groups = 5
group_sizes = int(len(df_here)/n_groups)
group_extra = len(df_here) - ((n_groups - 1) * group_sizes)

for g in range(n_groups):
    group_size = group_sizes if g < (n_groups - 1) else group_extra
    inds = np.random.choice(inds_available, size=group_size, replace=False)
    groups[inds] = g
    inds_available = list(set(inds_available) - set(inds))

df_here = df_here.with_columns(pl.Series('group', groups))

In [35]:
dict_probs = {}

for g in range(n_groups):
    df_sum = df_here.filter(df_here['group'] == g).sum()
    display(df_sum)

    d = {}
    d['n_less65'] = df_sum['age_less65'].to_numpy()[0]
    d['n_65'] = df_sum['age_65'].to_numpy()[0]
    d['n_70'] = df_sum['age_70'].to_numpy()[0]
    d['n_75'] = df_sum['age_75'].to_numpy()[0]
    d['n_over80'] = df_sum['age_over80'].to_numpy()[0]
    d['n_all'] = sum(list(d.values()))
    d['admissions'] = df_sum['admissions'].to_numpy()[0]

    dict_probs[g] = d

MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health,age_less65,age_65,age_70,age_75,age_over80,depriv_quantile_min,depriv_quantile_max,group
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,i64
null,3237.666667,11516.989331,2249629,null,1629007,323433,170521,207.552798,41.53531,21.911892,null,11.1995,10.0728,6.9836,232.7009,10.0411,2122961,1.8271e6,86648.7039,77545.6953,53811.7789,77832.7257,0.0,54.2,0


MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health,age_less65,age_65,age_70,age_75,age_over80,depriv_quantile_min,depriv_quantile_max,group
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,i64
null,3280.666667,11720.939689,2297232,null,1642516,327416,174767,206.966234,41.701509,22.332256,null,11.2071,10.3765,7.197,232.2072,10.015,2144699,1.8439e6,87282.488,80241.96,55573.1012,77685.3214,0.0,54.2,271


MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health,age_less65,age_65,age_70,age_75,age_over80,depriv_quantile_min,depriv_quantile_max,group
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,i64
null,3218.0,11679.89908,2286172,null,1638997,321776,172863,207.53623,41.256196,22.207575,null,11.0012,9.9663,6.9522,233.0814,9.9977,2133636,1.8423e6,84825.2647,76331.7623,53348.3384,76794.71,0.0,54.2,542


MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health,age_less65,age_65,age_70,age_75,age_over80,depriv_quantile_min,depriv_quantile_max,group
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,i64
null,3315.333333,11893.352009,2356059,null,1700783,333197,177112,207.689363,41.28293,22.027707,null,11.0455,9.9132,6.7531,233.8366,9.4521,2211092,1.9157e6,88113.623,78401.0089,53558.7469,75373.6418,0.0,54.2,813


MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health,age_less65,age_65,age_70,age_75,age_over80,depriv_quantile_min,depriv_quantile_max,group
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,i64
null,3368.666667,11776.374952,2285804,null,1651940,326362,173172,210.641647,42.054773,22.30358,null,11.0521,9.9753,6.9183,237.0448,10.0109,2151474,1.8604e6,85071.2184,76320.0903,52825.1211,76823.4365,0.0,55.0,1100


In [36]:
dict_probs

{0: {'n_less65': 1827099.1189000004,
  'n_65': 86648.70390000005,
  'n_70': 77545.69529999995,
  'n_75': 53811.77890000003,
  'n_over80': 77832.72570000001,
  'n_all': 2122938.0227000006,
  'admissions': 3237.666666675},
 1: {'n_less65': 1843936.788400001,
  'n_65': 87282.48800000001,
  'n_70': 80241.95999999998,
  'n_75': 55573.101199999976,
  'n_over80': 77685.32139999996,
  'n_all': 2144719.6590000005,
  'admissions': 3280.666666656999},
 2: {'n_less65': 1842322.6336000003,
  'n_65': 84825.26470000001,
  'n_70': 76331.76230000002,
  'n_75': 53348.33840000002,
  'n_over80': 76794.71000000005,
  'n_all': 2133622.7090000003,
  'admissions': 3218.000000006001},
 3: {'n_less65': 1915660.9886000012,
  'n_65': 88113.623,
  'n_70': 78401.00889999999,
  'n_75': 53558.746900000006,
  'n_over80': 75373.6418,
  'n_all': 2211108.009200001,
  'admissions': 3315.3333333340006},
 4: {'n_less65': 1860445.8472999996,
  'n_65': 85071.21840000001,
  'n_70': 76320.09029999998,
  'n_75': 52825.1211000000

In [42]:
def calculate_coeffs(d):
    
    # fractions:
    f_less65_65_0_1 = (d[1]['n_65'] / d[1]['n_less65']) - (d[0]['n_65'] / d[0]['n_less65'])
    f_less65_0_1 = (1.0 / d[1]['n_less65']) - (1.0 / d[0]['n_less65'])
    f_less65_70_1_0 = (d[0]['n_70'] / d[0]['n_less65']) - (d[1]['n_70'] / d[1]['n_less65'])
    f_less65_75_1_0 = (d[0]['n_75'] / d[0]['n_less65']) - (d[1]['n_75'] / d[1]['n_less65'])
    f_less65_over80_1_0 = (d[0]['n_over80'] / d[0]['n_less65']) - (d[1]['n_over80'] / d[1]['n_less65'])

    f_less65_65_2_3 = (d[3]['n_65'] / d[3]['n_less65']) - (d[2]['n_65'] / d[2]['n_less65'])
    f_less65_2_3 = (1.0 / d[3]['n_less65']) - (1.0 / d[2]['n_less65'])
    f_less65_70_3_2 = (d[2]['n_70'] / d[2]['n_less65']) - (d[3]['n_70'] / d[3]['n_less65'])
    f_less65_75_3_2 = (d[2]['n_75'] / d[2]['n_less65']) - (d[3]['n_75'] / d[3]['n_less65'])
    f_less65_over80_3_2 = (d[2]['n_over80'] / d[2]['n_less65']) - (d[3]['n_over80'] / d[3]['n_less65'])
    
    f_less65_65_3_4 = (d[4]['n_65'] / d[4]['n_less65']) - (d[3]['n_65'] / d[3]['n_less65'])
    f_less65_3_4 = (1.0 / d[4]['n_less65']) - (1.0 / d[3]['n_less65'])
    f_less65_70_4_3 = (d[3]['n_70'] / d[3]['n_less65']) - (d[4]['n_70'] / d[4]['n_less65'])
    f_less65_75_4_3 = (d[3]['n_75'] / d[3]['n_less65']) - (d[4]['n_75'] / d[4]['n_less65'])
    f_less65_over80_4_3 = (d[3]['n_over80'] / d[3]['n_less65']) - (d[4]['n_over80'] / d[4]['n_less65'])
    
    f_less65_65_0_2 = (d[2]['n_65'] / d[2]['n_less65']) - (d[0]['n_65'] / d[0]['n_less65'])
    f_less65_0_2 = (1.0 / d[2]['n_less65']) - (1.0 / d[0]['n_less65'])
    f_less65_70_2_0 = (d[0]['n_70'] / d[0]['n_less65']) - (d[2]['n_70'] / d[2]['n_less65'])
    f_less65_75_2_0 = (d[0]['n_75'] / d[0]['n_less65']) - (d[2]['n_75'] / d[2]['n_less65'])
    f_less65_over80_2_0 = (d[0]['n_over80'] / d[0]['n_less65']) - (d[2]['n_over80'] / d[2]['n_less65'])
    
    f_less65_65_1_4 = (d[4]['n_65'] / d[4]['n_less65']) - (d[1]['n_65'] / d[1]['n_less65'])
    f_less65_1_4 = (1.0 / d[4]['n_less65']) - (1.0 / d[1]['n_less65'])
    f_less65_70_4_1 = (d[1]['n_70'] / d[1]['n_less65']) - (d[4]['n_70'] / d[4]['n_less65'])
    f_less65_75_4_1 = (d[1]['n_75'] / d[1]['n_less65']) - (d[4]['n_75'] / d[4]['n_less65'])
    f_less65_over80_4_1 = (d[1]['n_over80'] / d[1]['n_less65']) - (d[4]['n_over80'] / d[4]['n_less65'])

    
    f_01 = f_less65_70_1_0 / f_less65_65_0_1
    f_23 = f_less65_70_3_2 / f_less65_65_2_3
    f_34 = f_less65_70_4_3 / f_less65_65_3_4
    f75_0123 = (f_less65_75_3_2 / f_less65_65_2_3) - (f_less65_75_1_0 / f_less65_65_0_1)
    f75_0134 = (f_less65_75_4_3 / f_less65_65_3_4) - (f_less65_75_1_0 / f_less65_65_0_1)
    f_lhs = (f75_0123 / (f_01 - f_23)) - (f75_0134 / (f_01 - f_34))


    f_02 = f_less65_70_2_0 / f_less65_65_0_2
    f_14 = f_less65_70_4_1 / f_less65_65_1_4
    f75_0214 = (f_less65_75_4_1 / f_less65_65_1_4) - (f_less65_75_2_0 / f_less65_65_0_2)
    f_lhs2 = (f75_0214 / (f_02 - f_14)) - (f75_0134 / (f_01 - f_34))


    lhs_1 = ((f_less65_over80_4_3/f_less65_65_3_4) - (f_less65_over80_1_0/f_less65_65_0_1)) / (f_01 - f_34)
    lhs_2 = ((f_less65_over80_3_2/f_less65_65_2_3) - (f_less65_over80_1_0/f_less65_65_0_1)) / (f_01 - f_23)
    lhs_3 = lhs_1  # ((f_less65_over80_4_3/f_less65_65_3_4) - (f_less65_over80_1_0/f_less65_65_0_1)) / (f_01 - f_34)
    lhs_4 = ((f_less65_over80_4_1/f_less65_65_1_4) - (f_less65_over80_2_0/f_less65_65_0_2)) / (f_02 - f_14)

    lhs = ((lhs_1 - lhs_2) / f_lhs) - ((lhs_3 - lhs_4) / f_lhs2)

    rhs_1 = (d[3]['admissions'] * (f_less65_3_4/f_less65_65_3_4) - d[0]['admissions'] * (f_less65_0_1/f_less65_65_0_1)) / (f_01 - f_34)
    rhs_2 = (d[2]['admissions'] * (f_less65_2_3/f_less65_65_2_3) - d[0]['admissions'] * (f_less65_0_1/f_less65_65_0_1)) / (f_01 - f_34)
    rhs_3 = rhs_1  #(d[3]['admissions'] * (f_less65_3_4/f_less65_65_3_4) - d[0]['admissions'] * (f_less65_0_1/f_less65_65_0_1)) / (f_01 - f_34)
    rhs_4 = (d[1]['admissions'] * (f_less65_1_4/f_less65_65_1_4) - d[0]['admissions'] * (f_less65_0_2/f_less65_65_0_2)) / (f_02 - f_14)
    
    rhs = -((rhs_1 - rhs_2) / f_lhs) + ((rhs_3 - rhs_4) / f_lhs2)
    
    c_over80 = rhs / lhs


    
    rhs_1 = (d[3]['admissions'] * f_less65_3_4 + c_over80 * f_less65_over80_4_3) / f_less65_65_3_4
    rhs_2 = (d[0]['admissions'] * f_less65_0_1 + c_over80 * f_less65_over80_1_0) / f_less65_65_0_1
    rhs_3 = (d[2]['admissions'] * f_less65_2_3 + c_over80 * f_less65_over80_3_2) / f_less65_65_2_3
    rhs_4 = rhs_2
    c_75 = (((rhs_1 - rhs_2) / (f_01 - f_34)) - ((rhs_3 - rhs_4) / (f_01 - f_23))) / f_lhs

    # coeff age 70-75:
    rhs_32 = (1.0 / f_less65_65_2_3) * (d[2]['admissions'] * f_less65_2_3 + c_75 * f_less65_75_3_2 + c_over80 * f_less65_over80_3_2)
    rhs_10 = (1.0 / f_less65_65_0_1) * (d[0]['admissions'] * f_less65_0_1 + c_75 * f_less65_75_1_0 + c_over80 * f_less65_over80_1_0)
    c_70 = 1.0 / ((f_less65_70_1_0 / f_less65_65_0_1) - (f_less65_70_3_2 / f_less65_65_2_3)) * (rhs_32 - rhs_10)

    # coeff age 65-70:
    c_65 = sum([
        d[0]['admissions'] * f_less65_0_1,
        c_70 * f_less65_70_1_0,
        c_75 * f_less65_75_1_0,
        c_over80 * f_less65_over80_1_0,
    ]) / f_less65_65_0_1

    # coeff age less than 65:
    c_less65 = sum([
        d[0]['admissions'],
        -c_65*(d[0]['n_65']),
        -c_70*(d[0]['n_70']),
        -c_75*(d[0]['n_75']),
        -c_over80*(d[0]['n_over80'])
    ]) / d[0]['n_less65']
    return (c_less65, c_65, c_70, c_75, c_over80)

In [44]:
c_less65_fit, c_65_fit, c_70_fit, c_75_fit, c_over80_fit = calculate_coeffs(dict_probs)

In [47]:
print(c_less65_fit, c_65_fit, c_70_fit, c_75_fit, c_over80_fit)

-0.005293094129963323 0.08086720499028623 0.11109029343742345 -0.1431895836665754 0.06414201627689652


In [46]:
for g, g_dict in dict_probs.items():
    print(g)
    new_sum = sum([
        c_less65_fit * g_dict['n_less65'],
        c_65_fit * g_dict['n_65'],
        c_70_fit * g_dict['n_70'],
        c_75_fit * g_dict['n_75'],
        c_over80_fit * g_dict['n_over80'],
    ])
    print(g_dict['admissions'], new_sum)

0
3237.666666675 3237.6666666750007
1
3280.666666656999 3237.6666666750007
2
3218.000000006001 2874.55399794609
3
3315.3333333340006 2860.8822531805254
4
3368.666666688001 2873.9809072502903
